# SmartFarm ML — Stage 02: Real classifiers (no leakage)
**Goal:** train genuine classifiers and understand the `fit` / `predict` loop.
This time the label is honest — it depends on soil, crop, temperature, time of day, AND noise,
so no single feature is the answer. Data file: `irrigation_honest.csv`.

## 1. Load

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("irrigation_honest.csv")
print(df.shape)
df.head()

(1500, 6)


,crop_type,growth_stage,soil_moisture,air_humidity,temperature,irrigate
0,tomato,flowering,56.7,58.8,31.7,0
1,okra,flowering,65.8,77.8,29.0,0
2,chili,vegetative,13.1,81.5,28.1,1
3,chili,vegetative,24.0,86.6,31.3,1
4,chili,seedling,32.1,66.2,30.8,0


## 2. Rules baseline FIRST — the number ML must beat
Before any ML, write the dumb-but-sensible rule and score it. If ML can't beat this,
ML isn't justified yet. This baseline is *crop-aware*: each crop has its own soil threshold.

In [6]:
# .map(dict) looks up each crop string and swaps in its threshold number (like a HashMap.get)
ideal = {"tomato": 58, "chili": 45, "okra": 50}

rule_pred = (df["soil_moisture"] < df["crop_type"].map(ideal)).astype(int)
baseline_acc = (rule_pred == df["irrigate"]).mean()

print(f"rules baseline accuracy: {baseline_acc:.3f}")

rules baseline accuracy: 0.778


## 3. Features & label, then split
The models get only the numeric sensors — NOT crop_type (encoding crops is Stage 05).
`stratify=y` keeps the same 0/1 ratio in both train and test, so the split is fair.

In [7]:
feature_cols = ["soil_moisture", "air_humidity", "temperature"]
X = df[feature_cols]
y = df["irrigate"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

## 4. Train three classifiers and compare train vs test
We print BOTH train and test accuracy on purpose — the gap between them tells us about overfitting.

In [16]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000),   # max_iter: give the solver room to converge
    "decision_tree_d3":    DecisionTreeClassifier(max_depth=3, random_state=0),
    "decision_tree_full":  DecisionTreeClassifier(random_state=0),      # no depth limit
    "random_forest":       RandomForestClassifier(n_estimators=200, random_state=0),
}

print(f"{'model':22s} {'train':>7s} {'test':>7s}")
for name, m in models.items():          # dict items -> (name, model) pairs
    m.fit(X_train, y_train)             # learn the rule from training data
    tr = m.score(X_train, y_train)      # accuracy on data it studied
    te = m.score(X_test,  y_test)       # accuracy on unseen data (the honest number)
    print(f"{name:22s} {tr:7.3f} {te:7.3f}")

print(f"\n{'rules_baseline':22s} {'':>7s} {baseline_acc:7.3f}")

model                    train    test
logistic_regression      0.777   0.782
decision_tree_d3         0.787   0.780
decision_tree_full       1.000   0.731
random_forest            1.000   0.762

rules_baseline                   0.778


## 5. Prove it's leakage, don't just trust the number
A decision tree learns threshold rules. Print the rules it found —
you'll literally see it split on `soil_moisture <= ~54`. It re-discovered our label formula.

In [24]:
from sklearn.tree import export_text

tree = models["decision_tree_d3"]          # dict-லிருந்து depth-3 tree-ஐ எடு
print(export_text(tree, feature_names=feature_cols))

|--- soil_moisture <= 48.75
|   |--- soil_moisture <= 35.75
|   |   |--- temperature <= 19.50
|   |   |   |--- class: 0
|   |   |--- temperature >  19.50
|   |   |   |--- class: 1
|   |--- soil_moisture >  35.75
|   |   |--- temperature <= 22.65
|   |   |   |--- class: 0
|   |   |--- temperature >  22.65
|   |   |   |--- class: 1
|--- soil_moisture >  48.75
|   |--- soil_moisture <= 61.65
|   |   |--- temperature <= 24.25
|   |   |   |--- class: 0
|   |   |--- temperature >  24.25
|   |   |   |--- class: 0
|   |--- soil_moisture >  61.65
|   |   |--- soil_moisture <= 66.65
|   |   |   |--- class: 0
|   |   |--- soil_moisture >  66.65
|   |   |   |--- class: 0



## What the numbers mean

**Logistic regression** — fits one smooth linear boundary. It *can't* memorise individual
rows, so `train ≈ test`. Stable, honest, no overfitting. Often the right first model for tabular data.

**Decision tree (depth 3)** — only 3 layers of if-else, so it stays simple. Behaves much like logreg here.

**Decision tree (full)** — no depth limit, so it keeps splitting until it memorises every
training row → `train = 1.000`. But that memorisation is noise, not pattern, so `test` DROPS.
This is **overfitting** — and note it's *different from Stage 01 leakage*:
- leakage = fake-perfect on the **test** set (answer hidden in a feature)
- overfitting = perfect on **train**, honestly worse on **test** (memorised the training data)

**Random forest** — many trees, each trained on random rows/features, then they vote.
The votes cancel out individual trees' memorised noise, so it generalises better than one full
tree (higher test), even though each tree still overfits (train = 1.000).

## The honest verdict
The best model barely ties the crop-aware rules baseline — and the fancy ones (full tree,
forest) actually LOSE to it on test. **So on these features, ML is not yet justified.**
Why? The models can't see `crop_type`, but the right threshold *differs per crop*. That missing
signal is exactly what Stage 05 adds. This is a valid, honest finding — not a failure.

## Your turn
1. In `models`, add `DecisionTreeClassifier(max_depth=5, random_state=0)`. Where does the
   train/test gap start opening up? That gap opening = overfitting beginning.
2. One sentence: why is `train = 1.000` here NOT the same red flag as the `1.000` in Stage 01?

In [18]:
lr = models["logistic_regression"]
for feat, w in zip(feature_cols, lr.coef_[0]):
    print(f"{feat}: {w:.3f}")

soil_moisture: -0.125
air_humidity: 0.009
temperature: 0.089


In [19]:
rf = models["random_forest"]
for feat, imp in zip(feature_cols, rf.feature_importances_):
    print(f"{feat}: {imp:.3f}")

soil_moisture: 0.558
air_humidity: 0.229
temperature: 0.213
